# Modele Rekurencyjne

In [1]:
import pickle
import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from utils import pad_collate, train_composer_classifier, evaluate_accuracy
from model import LSTMComposerClassifier

In [2]:
FOLDER_DIR = "pakiet/"

with open(FOLDER_DIR + "train.pkl", "rb") as f:
    train_data = pickle.load(f)

with open(FOLDER_DIR + "test_no_target.pkl", "rb") as f:
    test_data = pickle.load(f)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=pad_collate)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, collate_fn=pad_collate)

In [3]:
# Scan the loader to find the exact range of your raw chord data
absolute_min = 0
absolute_max = 0

for x_batch, _, _ in train_loader:
    b_min = x_batch.min().item()
    b_max = x_batch.max().item()
    if b_min < absolute_min: absolute_min = b_min
    if b_max > absolute_max: absolute_max = b_max

print(f"Raw data minimum index: {absolute_min}") # Probably -1
print(f"Raw data maximum index: {absolute_max}") # Probably 191


Raw data minimum index: 0
Raw data maximum index: 192


In [4]:
x_padded, lengths, y = next(iter(train_loader))

print("x_padded shape:", x_padded.shape)
print("lengths shape:", lengths.shape)
print("labels shape:", y.shape)

print("x_padded:", x_padded)

vocab_size = int(x_padded.max() + 2)
print("vocab_size:", vocab_size)

x_padded shape: torch.Size([32, 4016])
lengths shape: torch.Size([32])
labels shape: torch.Size([32])
x_padded: tensor([[  1,  13,   1,  ...,   0,   0,   0],
        [  6,  13,  13,  ...,   0,   0,   0],
        [ 67, 101, 186,  ...,   0,   0,   0],
        ...,
        [113,  35, 145,  ...,   0,   0,   0],
        [  0,   0,   0,  ...,   0,   0,   1],
        [ 74,  48,  48,  ...,   0,   0,   0]])
vocab_size: 194


In [5]:
VOCAB_SIZE = vocab_size 
EMBEDDING_DIM = 64
HIDDEN_SIZE = 128
NUM_LAYERS = 2
OUT_SIZE = 5

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = LSTMComposerClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS, OUT_SIZE).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
history_losses = []

In [7]:
train_composer_classifier(
    model=model,
    optimizer=optimizer,
    loader=train_loader,
    losses_list=history_losses,
    criterion=nn.CrossEntropyLoss(),
    device=device,
    epochs=5,
    show_every=1
)


[1/5], Loss: 1.2861
[2/5], Loss: 1.2667
[3/5], Loss: 1.2551
[4/5], Loss: 1.2440
[5/5], Loss: 1.2426


In [8]:
train_accuracy = evaluate_accuracy(model, train_loader, device)
print(f"\nFinal Training Accuracy: {train_accuracy:.2f}%")


Final Training Accuracy: 57.54%
